# Create MHLW Awards (厚生労働科学研究費補助金, MHLW GRANTS SYSTEM)

Creates Ministry of Health, Labour and Welfare (Japan) research-grant awards from
the MHLW GRANTS SYSTEM run by the National Institute of Public Health
(https://mhlw-grants.niph.go.jp/). The site's search has a CSV export
(`/search/csv?years=YYYY`, a Drupal views_data_export batch); the script pulls one
export per research fiscal year, FY1997..current. Each CSV row is one REPORT (an
annual 総括 report or a multi-year 総合 summary); the script aggregates them to
**one row per project (課題番号)**. **19,551 projects, start FY1995-2025 (reports FY1997-2025), 100% title/amount/dates/PI, JPY 656bn** (local run 2026-09-30). Japanese as published.

**Scope filter:** only grants whose 研究区分 grant name is MHLW research money
(厚生労働科学研究費補助金, its pre-2001 name 厚生科学研究費補助金,
厚生労働行政推進調査事業費補助金, and the late-1990s 特定疾患調査研究補助金 / 心身障害研究費補助金). The DB also hosts FY2023+ grants of the Children
and Families Agency (こども家庭科学研究費補助金 / こども家庭行政推進調査事業費補助金) and
FY2024+ grants of the Consumer Affairs Agency (食品衛生基準科学研究費補助金 / ...行政推進調査事業費補助金);
those are other funders and are dropped in the script.

**AMED overlap:** AMED (F4320311405, `CreateAMEDAwards`, provenance `amed_amedfind`,
priority 207) took over most MHLW *medical* R&D programmes from FY2015; those are AMED
grants (AMEDfind, `JP..`-prefixed numbers) and are NOT in this DB. The DB's FY1997-2014
records of the medical programmes (難治性疾患, がん臨床, ...) were MHLW grants at the
time and ship here under MHLW. The report counts drop from ~1,800/yr (FY2014) to ~700/yr
(FY2015+) accordingly. No project number in this DB has the AMED `JP` + 2-digit year +
2-letter form, so the two ingests do not share award ids.

**Prerequisites:**
- Run `scripts/local/mhlw_grants_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/mhlw_grants/mhlw_grants_projects.parquet` (§1.4 shrink guard).

**`funder_award_id` (§2.1.1):** the project number 課題番号 exactly as the DB publishes it.
From ~FY2019 it is `{FY2}{2 letters}{4 digits}` (e.g. `20FC1044`); citation-side check
(`openalex_awards_raw`, funder 4320321945): ~713 distinct shells carry this bare form vs
~199 `JPMH20FC1044` and ~32 `JP20FC1044`, so we ship the bare form; 628 existing shells collapse
onto these rows. FY2002-2018 numbers are Japanese (`H21-循環器等（生習）・一般-013`); citing
works romanise them inconsistently (`H29-Nanchi-Ippan-033`, `H29-Nanchitou(Nan)-Ippan-031`),
so those shells cannot collapse. Reports before ~FY2002 carry no project number ("-"): the
script keys them `MHLW-{start FY}-{sha1(title|PI)[:10]}` (column `project_no_synthetic` = true;
7,124 projects). Unnumbered early-year reports of projects that later got a number are folded
into the numbered project when (title, PI, start FY) matches exactly one.

**Amounts:** sum of the per-FY 研究費 (research funding, JPY) over the project's annual
reports in the DB (`amount_jpy`; per-FY breakdown kept in `annual_funding_json`). Projects
whose early years are missing from the DB are under-counted, never over-counted.
**Dates:** Japanese fiscal years: `start_date` = 開始年度-04-01, `end_date` =
(終了予定年度 + 1)-03-31 (planned end).
**People:** names are published family-first (`小川　俊夫`), split on the space; the PI's
affiliation is the trailing parenthetical. Co-investigators (研究分担者) from the latest
annual report go into `investigators`.

**Funder details (Path A, F4320* Crossref-registered):** funder_id `4320321945`,
metadata read from `openalex.funders.funders` (ror 03mwa7s65, doi 10.13039/501100003478).

**Priority:** `505` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.mhlw_grants_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/mhlw_grants/mhlw_grants_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.mhlw_grants_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.mhlw_grants_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.mhlw_grants_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320321945 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320321945;

## Step 2: Create MHLW Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.mhlw_grants_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320321945  -- Ministry of Health, Labour and Welfare
),

g AS (
    SELECT
        r.*,
        TRIM(r.project_no) AS award_no,
        TRY_CAST(r.start_fy AS INT) AS sfy,
        TRY_CAST(r.end_fy AS INT) AS efy,
        NULLIF(TRIM(r.lead_given_name), '') AS given,
        NULLIF(TRIM(r.lead_family_name), '') AS family,
        from_json(r.co_investigators_json,
                  'array<struct<name:string, given_name:string, family_name:string, former_name:string, affiliation:string>>') AS co
    FROM openalex.awards.mhlw_grants_raw r
    WHERE r.project_no IS NOT NULL AND TRIM(r.project_no) != ''
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.award_no)))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    NULLIF(TRIM(g.aim), '') as description,
    f.funder_id,
    g.award_no as funder_award_id,
    TRY_CAST(g.amount_jpy AS DOUBLE) as amount,
    CASE WHEN TRY_CAST(g.amount_jpy AS DOUBLE) IS NOT NULL THEN 'JPY' END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'research' as funding_type,
    NULLIF(TRIM(g.scheme), '') as funder_scheme,
    'mhlw_grants_niph' as provenance,
    CASE WHEN g.sfy IS NOT NULL THEN TRY_TO_DATE(CONCAT(g.sfy, '-04-01'), 'yyyy-MM-dd') END as start_date,
    CASE WHEN g.efy IS NOT NULL THEN TRY_TO_DATE(CONCAT(g.efy + 1, '-03-31'), 'yyyy-MM-dd') END as end_date,
    g.sfy as start_year,
    g.efy as end_year,
    -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
    CASE WHEN g.family IS NOT NULL THEN named_struct(
        'given_name', g.given,
        'family_name', g.family,
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', named_struct(
            'name', NULLIF(TRIM(g.lead_affiliation), ''),
            'country', CAST(NULL AS STRING),
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) END as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CASE WHEN size(filter(COALESCE(g.co, array()), x -> x.family_name IS NOT NULL)) > 0 THEN
        transform(filter(g.co, x -> x.family_name IS NOT NULL), x -> named_struct(
            'given_name', x.given_name,
            'family_name', x.family_name,
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(x.affiliation), ''),
                'country', CAST(NULL AS STRING),
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        ))
    END as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.award_no)))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM g
CROSS JOIN src_funder f;

In [ ]:
%sql
-- Shape check (§2.1.1): ids by form. Expect: new-form 課題番号 (FY2019+), Japanese Heisei-era
-- 課題番号, synthetic MHLW-{FY}-{hash} for pre-2002 reports; zero JPMH/JP-prefixed or empty ids.
SELECT
    CASE WHEN funder_award_id RLIKE '^[0-9]{2}[A-Z]{2}[0-9]{4}$' THEN 'new_form'
         WHEN funder_award_id RLIKE '^MHLW-(19|20)[0-9]{2}-[0-9a-f]{10}$' THEN 'synthetic'
         WHEN funder_award_id RLIKE '^(JPMH|JP)' THEN 'BAD_prefixed'
         WHEN funder_award_id IS NULL OR TRIM(funder_award_id) = '' THEN 'BAD_empty'
         ELSE 'native_other' END AS form,
    COUNT(*) AS n, MIN(start_year) AS min_sy, MAX(start_year) AS max_sy,
    COUNT(DISTINCT id) AS distinct_ids
FROM openalex.awards.mhlw_grants_awards
GROUP BY 1 ORDER BY 2 DESC;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'mhlw_grants_niph' AND priority = 505;

-- Insert into openalex_awards_raw with priority.
-- Priority 505: direct-from-funder ingest of the MHLW GRANTS SYSTEM (NIPH) CSV exports.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    505 as priority
FROM openalex.awards.mhlw_grants_awards;

## Verification Queries

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, start_year, end_year, amount, currency,
       lead_investigator.given_name, lead_investigator.family_name, lead_investigator.affiliation.name,
       size(investigators) AS n_co
FROM openalex.awards.mhlw_grants_awards LIMIT 20;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, ROUND(AVG(amount)) as avg_jpy
FROM openalex.awards.mhlw_grants_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.family_name AS family, lead_investigator.given_name AS given, COUNT(*) AS n
FROM openalex.awards.mhlw_grants_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.mhlw_grants_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.3 / 6.7 coverage.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    collect_set(currency) as currencies,
    MIN(amount) as min_amount, MAX(amount) as max_amount, AVG(amount) as avg_amount,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution,
    COUNT(investigators) as has_co_investigators
FROM openalex.awards.mhlw_grants_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'mhlw_grants_niph'
GROUP BY provenance, priority;